# 01 — Preparação de Dados para Fine-Tuning

**Módulo:** EAI_07 — IA Generativa  
**Submódulo:** 04_Fine_Tuning  
**Ambiente:** `eai07` (Python 3.11)

---

## O que você vai aprender

- **Synthetic data generation** — usar um LLM para gerar pares Q&A a partir de documentos existentes
- **Formato JSONL para fine-tuning** — estrutura exigida pela DeepSeek API (`messages` com system/user/assistant)
- **Validação de dataset** — verificar schema, tamanho de tokens e qualidade antes do upload
- **Split treino/validação** — separação estratificada por módulo para avaliação justa

---

### Estratégia: Synthetic Data Generation

O corpus já existe: os 26 `AGENT_CONTEXT.md` do curso (EAI_01 a EAI_08), com 1.553 chunks.  
Em vez de escrever exemplos manualmente, usamos o próprio LLM para gerar pares Q&A:

```
AGENT_CONTEXT.md  →  [LLM gera perguntas]  →  pares Q&A  →  JSONL  →  fine-tuning
```

Cada chunk vira **2-3 exemplos de treinamento** com perguntas variadas sobre o mesmo conteúdo.
O resultado é um modelo que responde perguntas técnicas sobre o curso com precisão e no estilo certo.

## Setup

In [9]:
import sys, os, json, time, random, re
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

sys.path.append(os.path.abspath('..'))
load_dotenv('../.env')

llm = OpenAI(
    api_key=os.getenv('DEEPSEEK_API_KEY'),
    base_url='https://api.deepseek.com'
)
LLM_MODEL    = os.getenv('LLM_MODEL', 'deepseek-chat')
PROJETO_BASE = os.path.abspath('../..')
DATA_DIR     = Path('../data/finetune')
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f'LLM     : {LLM_MODEL}')
print(f'Projeto : {PROJETO_BASE}')
print(f'Output  : {DATA_DIR.resolve()}')

LLM     : deepseek-chat
Projeto : C:\Users\Jorge Maques\Documents\Especialista_em_AI
Output  : C:\Users\Jorge Maques\Documents\Especialista_em_AI\EAI_07_AI_Generative\data\finetune


---
## 1. Coleta e Chunking do Corpus

Reutilizamos o chunking por seção dos notebooks de RAG.  
Cada chunk do `AGENT_CONTEXT.md` é uma unidade de conhecimento que vira exemplos de treino.

In [10]:
def chunk_por_secao(texto: str) -> list:
    """Divide AGENT_CONTEXT.md em seções por cabeçalho markdown."""
    chunks, titulo, linhas = [], 'Introdução', []
    for linha in texto.split('\n'):
        if linha.startswith('#'):
            if linhas:
                c = ' '.join(linhas).strip()
                if c:
                    chunks.append({'titulo': titulo, 'conteudo': c})
            titulo, linhas = linha.lstrip('#').strip(), []
        elif linha.strip():
            linhas.append(linha.strip())
    if linhas:
        c = ' '.join(linhas).strip()
        if c:
            chunks.append({'titulo': titulo, 'conteudo': c})
    return chunks


def encontrar_agent_contexts(pasta_raiz: str) -> list:
    """Retorna lista de (modulo, caminho) para todos os AGENT_CONTEXT.md."""
    encontrados, ignorar = [], {'.git', 'venv', '.venv', '__pycache__'}
    for raiz, dirs, arquivos in os.walk(pasta_raiz):
        dirs[:] = [d for d in dirs if d not in ignorar and not d.startswith('.')]
        if 'AGENT_CONTEXT.md' in arquivos:
            partes = raiz.replace('\\', '/').split('/')
            modulo = next((p for p in partes if p.startswith('EAI_')), os.path.basename(raiz))
            encontrados.append((modulo, os.path.join(raiz, 'AGENT_CONTEXT.md')))
    return sorted(encontrados)


def coletar_chunks(pasta_raiz: str) -> list:
    """Coleta todos os chunks de todos os AGENT_CONTEXT.md."""
    todos = []
    for modulo, caminho in encontrar_agent_contexts(pasta_raiz):
        with open(caminho, 'r', encoding='utf-8') as f:
            conteudo = f.read()
        for s in chunk_por_secao(conteudo):
            # Filtra seções muito curtas (menos de 50 caracteres) — pouco conteúdo para gerar Q&A
            if len(s['conteudo']) >= 50:
                todos.append({
                    'modulo'  : modulo,
                    'titulo'  : s['titulo'],
                    'conteudo': s['conteudo'],
                })
    return todos


CHUNKS = coletar_chunks(PROJETO_BASE)

print(f'Total de chunks coletados: {len(CHUNKS)}')
print()

# Distribuição por módulo
from collections import Counter
dist = Counter(c['modulo'] for c in CHUNKS)
print('Chunks por módulo:')
for modulo, n in sorted(dist.items()):
    print(f'  {modulo:45} {n:4d} chunks')

Total de chunks coletados: 1118

Chunks por módulo:
  EAI_01_Fundamentos_Matemática_para_IA           26 chunks
  EAI_02_Machine_Learning                        213 chunks
  EAI_03_Deep_Learning                           258 chunks
  EAI_04_NLP_Classico                            294 chunks
  EAI_05_NLP_com_Transformers                    104 chunks
  EAI_06_Visao_Computacional                     126 chunks
  EAI_07_AI_Generative                            73 chunks
  EAI_08_MLOps_e_Implantação                      24 chunks


---
## 2. Geração de Q&A Sintético

Para cada chunk, o LLM gera **3 perguntas** com respostas correspondentes.  
As perguntas são variadas em estilo:
- **Direta** — "O que é X?", "Como funciona Y?"
- **Aplicada** — "Quando usar X?", "Qual a diferença entre X e Y?"
- **Técnica** — "Qual o código para X?", "Quais os parâmetros de Y?"

O LLM retorna JSON estruturado para facilitar o parsing.

In [11]:
SYSTEM_GERADOR = """\
Você é um especialista em IA que cria material de estudo técnico de alta qualidade.
Dado um trecho de documentação técnica, gere exatamente 3 pares pergunta/resposta.

Regras:
- As perguntas devem ser variadas: uma direta, uma aplicada, uma técnica
- As respostas devem ser completas, precisas e baseadas APENAS no conteúdo fornecido
- Use linguagem técnica adequada para um desenvolvedor
- Responda APENAS com JSON válido, sem texto antes ou depois
- Formato obrigatório:

{"pares": [
  {"pergunta": "...", "resposta": "..."},
  {"pergunta": "...", "resposta": "..."},
  {"pergunta": "...", "resposta": "..."}
]}
"""


def gerar_qa(chunk: dict, max_tentativas: int = 3) -> list:
    """
    Gera pares Q&A para um chunk usando o LLM.
    Retorna lista de dicts {pergunta, resposta, modulo, titulo}.
    """
    prompt = f"""Módulo: {chunk['modulo']}
Seção: {chunk['titulo']}

Conteúdo:
{chunk['conteudo'][:1500]}

Gere 3 pares pergunta/resposta sobre este conteúdo."""

    for tentativa in range(max_tentativas):
        try:
            resp = llm.chat.completions.create(
                model       = LLM_MODEL,
                messages    = [
                    {'role': 'system', 'content': SYSTEM_GERADOR},
                    {'role': 'user',   'content': prompt},
                ],
                temperature = 0.7,
            )
            texto = resp.choices[0].message.content.strip()
            # Remove blocos de código markdown se presentes
            texto = re.sub(r'^```json\s*|^```\s*|\s*```$', '', texto, flags=re.MULTILINE).strip()
            dados = json.loads(texto)
            pares = []
            for par in dados.get('pares', []):
                if par.get('pergunta') and par.get('resposta'):
                    pares.append({
                        'pergunta': par['pergunta'].strip(),
                        'resposta': par['resposta'].strip(),
                        'modulo'  : chunk['modulo'],
                        'titulo'  : chunk['titulo'],
                    })
            return pares
        except (json.JSONDecodeError, KeyError) as e:
            if tentativa < max_tentativas - 1:
                time.sleep(1)
            else:
                print(f"  [ERRO] chunk '{chunk['titulo']}': {e}")
                return []


# Teste com 2 chunks antes de rodar tudo
print('Testando geração de Q&A com 2 chunks...\n')
amostras = random.sample(CHUNKS, 2)
for amostra in amostras:
    pares = gerar_qa(amostra)
    print(f"Módulo : {amostra['modulo']}")
    print(f"Seção  : {amostra['titulo']}")
    print(f"Gerado : {len(pares)} pares")
    for p in pares:
        print(f"  Q: {p['pergunta']}")
        print(f"  A: {p['resposta'][:120]}...")
    print()

Testando geração de Q&A com 2 chunks...

Módulo : EAI_06_Visao_Computacional
Seção  : Non-Maximum Suppression (NMS)
Gerado : 3 pares
  Q: Qual é o propósito principal da função manual_nms e qual parâmetro controla o nível de supressão?
  A: A função manual_nms implementa o algoritmo Non-Maximum Suppression (NMS) manualmente para eliminar caixas delimitadoras ...
  Q: Explique o passo a passo do algoritmo NMS implementado na função manual_nms, detalhando a lógica do loop while.
  A: O algoritmo na função manual_nms segue estes passos: 1. Ordena os índices das caixas por suas pontuações (scores) em ord...
  Q: Como a função compute_iou calcula a métrica Intersection over Union para uma caixa de referência (box1) contra um array de outras caixas (boxes)? Detalhe o cálculo das áreas de interseção e união.
  A: A função compute_iou calcula o IoU da seguinte forma: 1. Calcula as coordenadas da região de interseção entre box1 e cad...

Módulo : EAI_06_Visao_Computacional
Seção  : Solução 2: U

In [12]:
# Geração completa do dataset
# ⚠️  Isso faz ~1.553 chamadas à API — pode levar 20-40 minutos
# Para um dataset menor, ajuste MAX_CHUNKS abaixo

MAX_CHUNKS  = len(CHUNKS)   # use ex: 200 para teste rápido
DELAY_ENTRE = 0.3           # segundos entre chamadas (respeita rate limit)
CACHE_QA    = DATA_DIR / 'qa_gerado.jsonl'

# Retoma de onde parou se o cache existir
ja_processados = set()
todos_qa       = []

if CACHE_QA.exists():
    with open(CACHE_QA, 'r', encoding='utf-8') as f:
        for linha in f:
            qa = json.loads(linha)
            todos_qa.append(qa)
            ja_processados.add(f"{qa['modulo']}||{qa['titulo']}")
    print(f'Cache encontrado: {len(todos_qa)} pares já gerados. Retomando...')

chunks_pendentes = [
    c for c in CHUNKS[:MAX_CHUNKS]
    if f"{c['modulo']}||{c['titulo']}" not in ja_processados
]
print(f'Chunks pendentes: {len(chunks_pendentes)}')
print(f'Estimativa: ~{len(chunks_pendentes) * DELAY_ENTRE / 60:.0f}-{len(chunks_pendentes) * 2 / 60:.0f} minutos\n')

# Abre cache em modo append para salvar progressivamente
with open(CACHE_QA, 'a', encoding='utf-8') as f_cache:
    for i, chunk in enumerate(chunks_pendentes):
        pares = gerar_qa(chunk)
        for par in pares:
            f_cache.write(json.dumps(par, ensure_ascii=False) + '\n')
            todos_qa.append(par)

        if (i + 1) % 50 == 0:
            print(f'  {i+1}/{len(chunks_pendentes)} chunks | {len(todos_qa)} pares gerados')

        time.sleep(DELAY_ENTRE)

print(f'\nConcluído! Total de pares Q&A: {len(todos_qa)}')

Cache encontrado: 3345 pares já gerados. Retomando...
Chunks pendentes: 0
Estimativa: ~0-0 minutos


Concluído! Total de pares Q&A: 3345


---
## 3. Conversão para Formato JSONL de Fine-Tuning

A DeepSeek API (e OpenAI) exigem um formato específico para fine-tuning:

```json
{"messages": [
  {"role": "system",    "content": "<identidade do assistente>"},
  {"role": "user",      "content": "<pergunta>"},
  {"role": "assistant", "content": "<resposta esperada>"}
]}
```

Cada linha do arquivo `.jsonl` é um exemplo de treinamento independente.

In [13]:
SYSTEM_ASSISTENTE = """\
Você é o Assistente Técnico do curso Especialista em IA de Carlos Henrique.
Responde perguntas técnicas sobre os módulos EAI_01 a EAI_08, cobrindo:
matemática para IA, machine learning, deep learning, NLP, visão computacional,
IA generativa, MLOps e big data com PySpark.
Seja preciso, técnico e direto. Use exemplos de código quando relevante.
"""


def converter_para_jsonl_finetune(pares_qa: list) -> list:
    """Converte pares Q&A para o formato messages exigido pelo fine-tuning."""
    exemplos = []
    for par in pares_qa:
        exemplos.append({
            'messages': [
                {'role': 'system',    'content': SYSTEM_ASSISTENTE},
                {'role': 'user',      'content': par['pergunta']},
                {'role': 'assistant', 'content': par['resposta']},
            ]
        })
    return exemplos


DATASET = converter_para_jsonl_finetune(todos_qa)

print(f'Exemplos de treinamento: {len(DATASET)}')
print()
print('Exemplo de entrada de treinamento:')
print(json.dumps(DATASET[0], ensure_ascii=False, indent=2))

Exemplos de treinamento: 3345

Exemplo de entrada de treinamento:
{
  "messages": [
    {
      "role": "system",
      "content": "Você é o Assistente Técnico do curso Especialista em IA de Carlos Henrique.\nResponde perguntas técnicas sobre os módulos EAI_01 a EAI_08, cobrindo:\nmatemática para IA, machine learning, deep learning, NLP, visão computacional,\nIA generativa, MLOps e big data com PySpark.\nSeja preciso, técnico e direto. Use exemplos de código quando relevante.\n"
    },
    {
      "role": "user",
      "content": "Qual é o propósito principal do documento AGENT_CONTEXT.md no módulo EAI_01_Fundamentos_Matemática_para_IA?"
    },
    {
      "role": "assistant",
      "content": "O propósito principal do documento AGENT_CONTEXT.md é fornecer um contexto estruturado para que agentes de IA possam responder questões sobre o módulo EAI_01 Fundamentos Matemáticos."
    }
  ]
}


---
## 4. Validação do Dataset

Antes de enviar para a API, validamos:
- **Schema**: todas as mensagens têm `role` e `content`
- **Roles**: sequência correta (`system` → `user` → `assistant`)
- **Tamanho**: respostas não muito curtas (< 20 chars) nem longas demais (> 2000 chars)
- **Duplicatas**: perguntas idênticas no dataset

In [14]:
def validar_dataset(dataset: list) -> dict:
    """
    Valida o dataset antes do upload.
    Retorna relatório com erros e estatísticas.
    """
    erros         = []
    avisos        = []
    perguntas     = []
    lens_resposta = []

    for i, exemplo in enumerate(dataset):
        msgs = exemplo.get('messages', [])

        # Schema
        if not msgs:
            erros.append(f'[{i}] messages vazio')
            continue

        roles = [m.get('role') for m in msgs]

        # Roles obrigatórios
        if 'user' not in roles:
            erros.append(f'[{i}] sem role=user')
        if 'assistant' not in roles:
            erros.append(f'[{i}] sem role=assistant')

        # Conteúdo não vazio
        for m in msgs:
            if not m.get('content', '').strip():
                erros.append(f'[{i}] content vazio no role={m.get("role")}')

        # Tamanho da resposta
        assistente_msg = next((m for m in msgs if m['role'] == 'assistant'), None)
        user_msg       = next((m for m in msgs if m['role'] == 'user'), None)

        if assistente_msg:
            n = len(assistente_msg['content'])
            lens_resposta.append(n)
            if n < 20:
                avisos.append(f'[{i}] resposta muito curta ({n} chars)')
            if n > 2000:
                avisos.append(f'[{i}] resposta muito longa ({n} chars) — pode exceder token limit')

        if user_msg:
            perguntas.append(user_msg['content'])

    # Duplicatas
    duplicatas = len(perguntas) - len(set(perguntas))
    if duplicatas > 0:
        avisos.append(f'{duplicatas} perguntas duplicadas no dataset')

    return {
        'total'          : len(dataset),
        'erros'          : erros,
        'avisos'         : avisos,
        'valido'         : len(erros) == 0,
        'resp_media'     : round(sum(lens_resposta) / len(lens_resposta), 0) if lens_resposta else 0,
        'resp_min'       : min(lens_resposta) if lens_resposta else 0,
        'resp_max'       : max(lens_resposta) if lens_resposta else 0,
        'duplicatas'     : duplicatas,
    }


relatorio = validar_dataset(DATASET)

print(f'Dataset válido : {relatorio["valido"]}')
print(f'Total exemplos : {relatorio["total"]}')
print(f'Duplicatas     : {relatorio["duplicatas"]}')
print(f'Resp. média    : {relatorio["resp_media"]:.0f} chars')
print(f'Resp. min/max  : {relatorio["resp_min"]} / {relatorio["resp_max"]} chars')

if relatorio['erros']:
    print(f'\n❌ Erros ({len(relatorio["erros"])}):')
    for e in relatorio['erros'][:10]:
        print(f'  {e}')
else:
    print('\n✅ Nenhum erro encontrado')

if relatorio['avisos']:
    print(f'\n⚠️  Avisos ({len(relatorio["avisos"])}):')
    for a in relatorio['avisos'][:10]:
        print(f'  {a}')

Dataset válido : True
Total exemplos : 3345
Duplicatas     : 18
Resp. média    : 391 chars
Resp. min/max  : 46 / 1268 chars

✅ Nenhum erro encontrado

⚠️  Avisos (1):
  18 perguntas duplicadas no dataset


---
## 5. Split Treino / Validação

Separamos **90% treino / 10% validação** com estratificação por módulo.  
Isso garante que todos os módulos do curso aparecem nos dois splits —
a avaliação reflete performance em todo o curso, não só nos módulos mais frequentes.

In [15]:
from collections import defaultdict

def split_estratificado(dataset: list, qa_raw: list, val_ratio: float = 0.1, seed: int = 42) -> tuple:
    """
    Split treino/validação estratificado por módulo.
    Garante que cada módulo está representado nos dois splits.

    Retorna (treino, validacao).
    """
    random.seed(seed)

    # Agrupa índices por módulo usando os metadados do qa_raw
    por_modulo = defaultdict(list)
    for i, qa in enumerate(qa_raw):
        por_modulo[qa['modulo']].append(i)

    treino_idx, val_idx = [], []

    for modulo, indices in por_modulo.items():
        random.shuffle(indices)
        n_val = max(1, int(len(indices) * val_ratio))
        val_idx.extend(indices[:n_val])
        treino_idx.extend(indices[n_val:])

    treino = [dataset[i] for i in treino_idx]
    val    = [dataset[i] for i in val_idx]

    return treino, val


TREINO, VAL = split_estratificado(DATASET, todos_qa, val_ratio=0.1)

print(f'Split realizado:')
print(f'  Treino    : {len(TREINO)} exemplos ({len(TREINO)/len(DATASET)*100:.0f}%)')
print(f'  Validação : {len(VAL)} exemplos ({len(VAL)/len(DATASET)*100:.0f}%)')

# Verificação da estratificação — cada módulo deve aparecer nos dois splits
modulos_treino = set(todos_qa[DATASET.index(e)]['modulo'] for e in TREINO if e in DATASET)
print(f'\nMódulos no treino    : {len(modulos_treino)}')

Split realizado:
  Treino    : 3015 exemplos (90%)
  Validação : 330 exemplos (10%)

Módulos no treino    : 8


---
## 6. Salvar Arquivos JSONL

In [17]:
def _testa_json(linha: str) -> bool:
    try:
        json.loads(linha)
        return True
    except json.JSONDecodeError:
        return False


def salvar_jsonl(dados: list, caminho: Path):
    """Salva lista de dicts em formato JSONL (uma linha por exemplo)."""
    with open(caminho, "w", encoding="utf-8") as f:
        for exemplo in dados:
            f.write(json.dumps(exemplo, ensure_ascii=False) + " ")
    tamanho = caminho.stat().st_size / 1024
    print(f"Salvo: {caminho} ({len(dados)} exemplos, {tamanho:.1f} KB)")


TRAIN_PATH = DATA_DIR / "train.jsonl"
VAL_PATH   = DATA_DIR / "val.jsonl"

salvar_jsonl(TREINO, TRAIN_PATH)
salvar_jsonl(VAL,    VAL_PATH)

print()
print("Verificação final dos arquivos JSONL:")
for path in [TRAIN_PATH, VAL_PATH]:
    with open(path, "r", encoding="utf-8") as f:
        linhas = f.readlines()
    erros_json = sum(1 for l in linhas if not _testa_json(l))
    status = "✅" if erros_json == 0 else "❌"
    print(f"  {status} {path.name}: {len(linhas)} linhas, {erros_json} erros de JSON")

print()
print("Arquivos prontos para upload:")
print(f"  Treino    : {TRAIN_PATH}")
print(f"  Validação : {VAL_PATH}")
print()
print("Próximo passo: 02_fine_tuning_deepseek.ipynb")


Salvo: ..\data\finetune\train.jsonl (3015 exemplos, 3025.5 KB)
Salvo: ..\data\finetune\val.jsonl (330 exemplos, 329.7 KB)

Verificação final dos arquivos JSONL:
  ❌ train.jsonl: 1 linhas, 1 erros de JSON
  ❌ val.jsonl: 1 linhas, 1 erros de JSON

Arquivos prontos para upload:
  Treino    : ..\data\finetune\train.jsonl
  Validação : ..\data\finetune\val.jsonl

Próximo passo: 02_fine_tuning_deepseek.ipynb


---
## Resumo

| Etapa | O que faz |
|---|---|
| **Coleta de corpus** | Lê 26 AGENT_CONTEXT.md → chunks por seção (≥ 50 chars) |
| **Synthetic data** | LLM gera 3 pares Q&A por chunk com retry automático |
| **Cache progressivo** | Salva em JSONL linha a linha — retoma se interrompido |
| **Formato fine-tuning** | Converte para `messages` (system / user / assistant) |
| **Validação** | Schema, roles, tamanho de resposta, duplicatas |
| **Split estratificado** | 90/10 com todos os módulos em ambos os splits |

### Arquivos gerados

```
data/finetune/
├── qa_gerado.jsonl   ← cache bruto dos pares Q&A (com metadados)
├── train.jsonl       ← dataset de treino no formato fine-tuning
└── val.jsonl         ← dataset de validação no formato fine-tuning
```